## PCA, SVD and Autoencoders

In this notebook we apply PCA to a OIS swap data from the Bank of England. The data consists of continuously compounded spot yields quoted for maturities out to 5 years. Data from 2009 until 2021 is included.

The notebook will apply traditional PCA methodology to the data and the use an SVD return to perform the same calculation. Finally a linear autoencoder is trained on the same dataset and shown to replicate SVD/PCA.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import trange

### Prepare the data

Firstly download the dataset if required

In [ ]:
import requests, zipfile, io
zip_file_url = "https://www.bankofengland.co.uk/-/media/boe/files/statistics/yield-curves/oisddata.zip"

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

with requests.get(zip_file_url, headers=headers, stream=True) as r:
    r.raise_for_status()
    with open("oisddata.zip", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

zip_path = "oisddata.zip"
extract_path = ""

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

The data is supplied in two Excel files, covering 2009-2015 and 2016 onwards.

In [ ]:
df1 = pd.read_excel("OIS daily data_2009 to 2015.xlsx", 
                    index_col=0, header=3, sheet_name="2. spot curve", skiprows=[4],
                    engine='openpyxl')

In [ ]:
df1.dropna(inplace=True)

In [ ]:
df2 = pd.read_excel("OIS daily data_2016 to 2024.xlsx", 
                    index_col=0, header=3, sheet_name="3. spot, short end", skiprows=[4],
                    engine='openpyxl')

In [ ]:
df2.dropna(inplace=True)

In [ ]:
df2[:1768]

The data is limited to the end of 2022

In [ ]:
df = pd.concat([df1, df2[:1768]])

A 3D plot of the data

In [ ]:
x = df.columns
dates_formatted = pd.to_datetime(df.index)
y = [ (d-min(dates_formatted)).days for d in dates_formatted]
X,Y = np.meshgrid(x,y)
Z = df

In [ ]:
fig = plt.figure(figsize=[10,10])
ax = plt.axes(projection="3d")
ax.plot_surface(X, Y, Z, cmap=plt.cm.viridis, linewidth=0.2)
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Date (days from 02/01/2009)")
ax.set_zlabel("Rate")
plt.show()

In [ ]:
X = df.to_numpy()

Apply standard scaling (note this differs slightly from the text in that data is also adjusted for the variance).

In [ ]:
scaler = StandardScaler()

In [ ]:
X_bar = scaler.fit_transform(X)

### Classic PCA

Calculate the correlation matrix

In [ ]:
rho = np.corrcoef(X_bar.T)

In [ ]:
print(rho)

Calculate the eigenvalues and eigenvectors using numpy.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(rho)
df_eigval = pd.DataFrame({"Eigenvalues":np.real(eigenvalues)})

Display the first 10 of 60 eigenvalues.

In [ ]:
df_eigval["Explained variance"] = df_eigval["Eigenvalues"] / np.sum(df_eigval["Eigenvalues"])
df_eigval[:10].style.format({"Explained variance": "{:.2%}"})

In [ ]:
print(sum(df_eigval["Explained variance"][:4]))

The first four eigenvalues explain 99.9% of the variance, henve we take the first 4 eigenvectors only and then reconstruct the data.

In [ ]:
ZL = np.dot(X_bar, np.real(eigenvectors[:,:4]))

In [ ]:
X_hat_bar = np.dot(ZL, np.real(eigenvectors[:,:4].T))

In [ ]:
X_hat = scaler.inverse_transform(X_hat_bar)

Now we calculate the reconstruction error relative to the norm of the data.

In [ ]:
np.linalg.norm(X_hat - X) / np.linalg.norm(X)

### SVD

The Singular Value Decomposition of the scaled matrix is calculated using the SVD routine from numpy.

In [ ]:
U, S, Vh = np.linalg.svd(X_bar, full_matrices=False)

We just want to first 4 singular values for the reconstuction so we extract these and build the full $\hat{S}$ matrix.

In [ ]:
import copy
S_hatd = copy.deepcopy(S)
S_hatd[4:] = 0.0
S_hat = np.zeros((len(S_hatd), len(S_hatd)))
np.fill_diagonal(S_hat, S_hatd)

In [ ]:
print(S_hat)

Reconstruct the data

In [ ]:
X_hatsvd_bar = np.dot(U, np.dot(S_hat, Vh))
X_hatsvd = scaler.inverse_transform(X_hatsvd_bar)

Calculate the relative reconstruction error.

In [ ]:
np.linalg.norm(X_hatsvd - X) / np.linalg.norm(X)

Which replaces the result from the classic PCA to 16 decimal places.

### Linear Autoencoder

We now construct a linear autoencoder with a single hidden layer containing four units. However, we need to be able to tie the weights between encoder and decoder so we define a class for this purpose.

In [ ]:
class DenseTied(nn.Module):
    def __init__(self, dense, activation=None, use_bias=True):
        super(DenseTied, self).__init__()
        self.dense = dense
        self.activation = activation if activation is not None else nn.Identity()
        self.use_bias = use_bias
        if self.use_bias:
            self.biases = nn.Parameter(torch.zeros(dense.in_features))

    def forward(self, inputs):
        z = torch.matmul(inputs, self.dense.weight)
        if self.use_bias:
            outputs = self.activation(z + self.biases)
        else:
            outputs = self.activation(z)
        return outputs

We define a function to build the model and then train it

In [ ]:
class LinearAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_width, activation=None, use_bias=True):
        super(LinearAutoencoder, self).__init__()
        self.dense = nn.Linear(input_dim, hidden_width, bias=False)
        self.tied_dense = DenseTied(self.dense, activation=activation, use_bias=use_bias)

    def forward(self, x):
        encoded = self.dense(x)
        decoded = self.tied_dense(encoded)
        return decoded

def build_linear_autoencoder(input_dim, hidden_width):
    model = LinearAutoencoder(input_dim, hidden_width, activation=nn.Identity(), use_bias=False)
    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
input_dim = 60
hidden_width = 4
no_epochs = 5000
batch_size = 512

In [ ]:
model = build_linear_autoencoder(input_dim, hidden_width).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
tX_bar = torch.Tensor(X_bar).to(device)

dataset = torch.utils.data.TensorDataset(tX_bar, tX_bar)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
model.train()
for epoch in trange(no_epochs):
    for batch in dataloader:
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    if epoch % 100 == 0:  # Print loss every 100 epochs
        print(f'Epoch {epoch}/{no_epochs}, Loss: {loss.item()}')

Finally we unscale and calculate the reconstruction error

In [ ]:
X_hat_auto_bar = model(tX_bar).cpu().detach().numpy()

In [ ]:
X_hat_auto = scaler.inverse_transform(X_hat_auto_bar)

In [ ]:
np.linalg.norm(X_hat_auto - X) / np.linalg.norm(X)

### Comparing reconstructed curves

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['--', ':', '-.']))
ax.set_prop_cycle(monochrome)

ax.plot(x, X_hat[0,:] - df.iloc[0,:],  alpha=0.5, label = 'PCA')
ax.plot(x, X_hatsvd[0,:] - df.iloc[0,:],  alpha=0.5, label = 'SVD', marker = '+')
ax.plot(x, X_hat_auto[0,:] - df.iloc[0,:],  alpha=0.5, label = 'Linear')   
ax.set_xlabel('Maturity')
ax.set_ylabel('Rate Error (%)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('PCAReconstruct.png', dpi=300, bbox_inches='tight')